### code for comparing quality of automatic and manual transcriptions

### imports

In [1]:
import numpy as np
import pandas as pd
import hypertools as hyp
import quail
import pickle
import random
import re
import os
import json
from num2words import num2words
from nltk.corpus import stopwords
from scipy.signal import resample
from scipy.stats import pearsonr, sem
from scipy.spatial.distance import cdist

import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

### paths

In [2]:
data_dir = '../../../data/'
auto_dir = data_dir+'transcriptions/automatic/'
man_dir = data_dir+'transcriptions/manual/'

### data

In [3]:
with open(data_dir+'pickles/id_maps.p', 'rb') as f:
    id_maps = pickle.load(f)

In [4]:
def load_transcript(path):

    try:
        with open(path, 'r') as f:
            transcript = f.read()
        # format automatic transcripts
        if '.wav' in path:
            transcript = ' '.join([line.split(',')[0].lower() for line in transcript.split('\n')])
        else:
            transcript = transcript.lower() 
        return transcript

    except FileNotFoundError:
        print(f'No transcript at {path}')
        return

In [5]:
autos = dict.fromkeys(id_maps.keys())
mans = dict.fromkeys(id_maps.keys())

for sid, maps in id_maps.items():
    tid1 = maps['session 1']
    tid2 = maps['session 2']
    
    for method_dict, root in zip([autos, mans], [auto_dir, man_dir]):
        ext = '.txt'
        if 'automatic' in root:
            ext = f'-corrected.wav{ext}'
            
        rec1_path = os.path.join(root, sid, tid1, f'{tid1}-recall{ext}')
        pred_path = os.path.join(root, sid, tid1, f'{tid1}-prediction{ext}')
        del_path = os.path.join(root, sid, tid2, f'{tid2}-delayed{ext}')
        rec2_path = os.path.join(root, sid, tid2, f'{tid2}-recall{ext}')

        zipit = zip(['rec1', 'prediction', 'delayed', 'rec2'], [rec1_path, pred_path, del_path, rec2_path])

        method_dict[sid] = {rectype : load_transcript(path) for rectype, path in zipit}

No transcript at ../../../data/transcriptions/manual/MD-101218-A-01/debugIEH2T:debugDLVLJ/debugIEH2T:debugDLVLJ-recall.txt
No transcript at ../../../data/transcriptions/manual/MD-101218-A-01/debugIEH2T:debugDLVLJ/debugIEH2T:debugDLVLJ-prediction.txt
No transcript at ../../../data/transcriptions/manual/MD-101218-A-01/debug2Ea7T:debugosNZ7/debug2Ea7T:debugosNZ7-delayed.txt
No transcript at ../../../data/transcriptions/manual/MD-101218-A-01/debug2Ea7T:debugosNZ7/debug2Ea7T:debugosNZ7-recall.txt
No transcript at ../../../data/transcriptions/manual/MD-101218-B-01/debugBUnNA:debugLtZcs/debugBUnNA:debugLtZcs-recall.txt
No transcript at ../../../data/transcriptions/manual/MD-101218-B-01/debugBUnNA:debugLtZcs/debugBUnNA:debugLtZcs-prediction.txt
No transcript at ../../../data/transcriptions/manual/MD-101218-B-01/debugQEynG:debugpwxCU/debugQEynG:debugpwxCU-delayed.txt
No transcript at ../../../data/transcriptions/manual/MD-101218-B-01/debugQEynG:debugpwxCU/debugQEynG:debugpwxCU-recall.txt
No tra

In [10]:
# check that both datasets contain all transcripts
for sid in autos.keys():
    if ((all(autos[sid].values()) and not all(mans[sid].values()))
        or (all(mans[sid].values()) and not all(autos[sid].values()))):
        
        print(sid)

MD-101218-A-01
MD-101218-B-01
MD-101218-A-02
MD-101218-B-02
MD-101218-A-03
MD-101218-B-03
MD-101218-A-04
MD-101318-B-01
MD-101318-B-02
MD-101318-A-02
MD-101318-A-03
MD-101318-B-03
MD-101318-A-04
MD-101318-B-04
MD-101318-B-05
MD-101318-A-06
MD-101318-B-06
MD-101618-A-01
MD-101618-B-01
MD-101618-A-02
MD-101618-B-02
MD-101618-A-03
MD-101618-B-03
MD-101618-A-04
MD-101618-B-04
MD-102018-A-01
MD-102018-B-01
MD-102018-A-02
MD-102018-B-02
MD-102018-A-03
MD-102118-A-01
MD-102218-A-01
MD-102218-B-02
MD-102218-A-02
MD-102218-B-03
MD-102218-B-05
MD-102218-A-05
MD-102218-A-06
MD-102218-B-07
MD-020819-B-01
MD-021519-A-01
MD-021819-B-01
MD-021819-B-02
MD-021919-A-01
MD-021919-B-01
MD-021919-A-02
MD-022519-A-01
MD-022719-A-01
MD-022819-B-01
MD-022819-B-02


{'MD-013119-A-02': {'delayed': None,
  'prediction': None,
  'rec1': "so the episode starts with paper boi and earn darius skinny friends and some altercation at this man and woman at some kind of store or gas station is escalating paper boy pulls out his gun and holds it up to the man earn sees this dog and which makes him question things and i don't really know until why paper boy man and this is kind of like in a flash forward moment and then it goes to the kind of i guess it just it goes to earn his ex girlfriend in bed and he's your counting a dream and it's very clear that the woman i think it's bad is jealous of the woman in his dream and they're kind of joking about it and then she asks him to tell her but he loves her and she kind of brushes that off this is why i like they're kissing and she she gets frustrated so clear shoes jealous but then she gets up to fix her hair when she goes to their child and she reminds him that it's his night to take care of her and she tells him 

In [21]:
with open(man_dir+'MD-020119-A-03/debugRQQWb:debugwDi2H/debugRQQWb:debugwDi2H-recall.txt', 'r') as f:
    mtest3 = f.read()
    
with open(auto_dir+'MD-020119-A-03/debugRQQWb:debugwDi2H/debugRQQWb:debugwDi2H-recall-corrected.wav.txt', 'r') as f:
    atest3 = f.read()
    
# atest1 = ' '.join([line.split(',')[0].lower() for line in atest1.split('\n')])
# atest2 = ' '.join([line.split(',')[0].lower() for line in atest2.split('\n')])
atest3 = ' '.join([line.split(',')[0].lower() for line in atest3.split('\n')])

In [55]:
len(mtest1.split())

2569

In [56]:
len(atest1.split())

1174

In [57]:
len(mtest2.split())

2495

In [58]:
len(atest2.split())

1683

In [64]:
len(mtest3.split())

2514

In [65]:
len(atest3.split())

2027

In [24]:
mtest3.lower()

"so the episode began with earn alfred and darius running out of the car and they were running to this guy because it didn't show it at the time but he had like a broken car mirror so he was alfred was asking money for it and then earn was trying to tell him to stop and then darius was really high and so he was kinda like looking around and he saw a dog and was like watching the dog and then there was the guy who had broken the mirror also had a girl and the girl recognized alfred as paper boy and then and then the guy who had broken the mirror was pretended to recognize him but also just trashed his music and then alfred got angry because of that or alfred got angry because he wasn't getting money and was like trashing his name so he like pulled out a gun and then and then earn also had a gun on him and then darius also had a gun on him and was saying that his guys are like nearby so he should just so alfred should stop but alfred still cocked the gun and like put it to his chest and 

In [23]:
atest3

"the episode began with darius running out of the car and they're running to the sky because it isn't shot the time but he had like he had a broken car mirror so is that i was asking money for it and then earned was trying to tell him to stop and then darius was really high and so he was while i look around and he saw a dog and was like watching the dog man there is a guy that had broken her also had a girl with him and the girl recognized offers paper boy and then and then the guy who had broken the mirror pretended to recognise him also like just trashes music and then offered got angry because of that angry and then and then earn also had a gun on him that darius also had a gun on him as saying that his guys are nearby so he should fridge stopped but i'll friend still octagon and like put it to his chest and then eminem and then you like each other and then it blacked out and then it was the theme song of episode and then it went to earn when i think it went to earn waking up next t

In [70]:
with open(auto_dir+'MD-020719-B-01/debugbICiy:debugKisj0/debugbICiy:debugKisj0-delayed-corrected.wav.txt', 'r') as f:
    test = f.read()

In [71]:
test = ' '.join([line.split(',')[0].lower() for line in test.split('\n')])

In [72]:
test

"so that's okay i really don't remember anyone's name for paper boy seven episode last week started kiss in the parking lot where the mirror had been in the car which we find out later with damian with his like sidekick and it was a man or woman they kicked off paper boys mirror and then they all got out and they were arguing and then he finds out that the guy who is the main character like i've so much deja vu right now his sidekick is really high but she sees the dog the plunge the creepy line train to try to give him the nutella sandwich is the dog run into the wall and then they end up shooting the guy at the episode then goes into the main character in the woman has child with they're in bed together and he's listen to music really loud headphones and she's like why you awake you said that you can sleep as he had a really bad dream and it was about check out the dream was sir kissing in bed and then she says what tell me you love me or something and he doesn't already loves her wh